# EEI6366 Lab 01: Big Data Processing with PySpark

**Team:** Eastern Developers  
**Dataset:** `open_university_student_performance.csv`  
**Size:** 150,500 rows  

This notebook covers:
- Loading and cleaning the dataset (missing values, duplicates, outliers)
- **PySpark**: 4 easy operations + visualizations
- **Spark SQL**: 4 easy operations + visualizations

## 1. Install and Start PySpark

In [ ]:
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder.appName("Lab01-Student-Performance").getOrCreate()
spark

## 2. Load the Dataset

In [ ]:
from google.colab import files
print("Upload open_university_student_performance.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Dataset loaded successfully")
print("Total rows:", df.count())
df.show(5)

## 3. Explore Dataset Structure

In [ ]:
# Show column names and data types
df.printSchema()

print("Number of columns:", len(df.columns))
print("Column names:", df.columns)

## 4. Check Missing Values

In [ ]:
# Count null values in each column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## 5. Check and Remove Duplicates

In [ ]:
# Check how many duplicate rows exist
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

# Remove duplicates
df = df.dropDuplicates()
print("After removing duplicates:", df.count())

## 6. Handle Missing Values

In [ ]:
# Drop rows that have any missing values
df = df.dropna()

print("Rows after dropping missing values:", df.count())

## 7. Find and Show Outliers (Overall Mark)

We use the simple **IQR method** to find outliers in `overall_mark`.

In [ ]:
# Calculate Q1, Q3 and IQR for overall_mark
quantiles = df.approxQuantile("overall_mark", [0.25, 0.75], 0.05)
Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

# Find outlier rows
outliers = df.filter(
    (F.col("overall_mark") < lower_bound) | (F.col("overall_mark") > upper_bound)
)

print("\nNumber of outliers:", outliers.count())
outliers.select("student_id", "course_code", "overall_mark", "grade").show(10)

In [ ]:
# Boxplot to visualize outliers
pdf = df.select("overall_mark").toPandas()

plt.figure(figsize=(8, 5))
plt.boxplot(pdf["overall_mark"].dropna(), vert=True)
plt.title("Boxplot of Overall Mark (Outliers)")
plt.ylabel("Overall Mark")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 8. Verify Cleaned Dataset

In [ ]:
print("Final number of rows:", df.count())
print("Final number of columns:", len(df.columns))

# Check again for missing values (should be 0)
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

df.show(5)

## 9. PySpark Operations (4 Easy Operations)

### Operation 1 – Show Schema and Count

See the structure of the cleaned data and total number of rows.

In [ ]:
df.printSchema()
print("Total rows:", df.count())

### Operation 2 – Filter Data

Show only the students who **Failed**.

In [ ]:
failed_students = df.filter(df.status == "Fail")
failed_students.show(10)
print("Number of Fail students:", failed_students.count())

### Operation 3 – Select Columns and Order

Select only a few columns and sort by overall mark (highest first).

In [ ]:
df.select("student_id", "course_code", "overall_mark", "grade") \
  .orderBy(F.desc("overall_mark")) \
  .show(10)

### Operation 4 – GroupBy and Average + Visualization

Find the average overall mark for each center and show it as a bar chart.

In [ ]:
center_avg = df.groupBy("center").avg("overall_mark")
center_avg.show()

# Convert to Pandas for plotting
center_pdf = center_avg.toPandas()
center_pdf = center_pdf.sort_values("avg(overall_mark)", ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(center_pdf["center"], center_pdf["avg(overall_mark)"], color="steelblue")
plt.title("Average Overall Mark by Center")
plt.xlabel("Center")
plt.ylabel("Average Overall Mark")
plt.xticks(rotation=30)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 10. Spark SQL Operations (4 Easy Operations)

First create a temporary view so we can use SQL.

In [ ]:
df.createOrReplaceTempView("students")

### SQL Operation 1 – Simple SELECT with WHERE

Show students from Colombo center who got grade A or A+.

In [ ]:
spark.sql("""
SELECT student_id, course_code, overall_mark, grade
FROM students
WHERE center = 'Colombo' AND grade IN ('A', 'A+')
LIMIT 10
""").show()

### SQL Operation 2 – COUNT by Status + Visualization

Count how many students Passed and Failed, then show a pie chart.

In [ ]:
status_count = spark.sql("""
SELECT status, COUNT(*) AS total
FROM students
GROUP BY status
""")
status_count.show()

# Pie chart
status_pdf = status_count.toPandas()

plt.figure(figsize=(6, 6))
plt.pie(status_pdf["total"], labels=status_pdf["status"], autopct="%1.1f%%",
        colors=["#4CAF50", "#F44336"], startangle=90)
plt.title("Pass vs Fail Distribution")
plt.show()

### SQL Operation 3 – Average Mark by Course + Visualization

Find the average overall mark for each course and show a bar chart.

In [ ]:
course_avg = spark.sql("""
SELECT course_code, ROUND(AVG(overall_mark), 2) AS avg_mark
FROM students
GROUP BY course_code
ORDER BY avg_mark DESC
""")
course_avg.show(truncate=False)

# Bar chart
course_pdf = course_avg.toPandas()

plt.figure(figsize=(10, 5))
plt.barh(course_pdf["course_code"], course_pdf["avg_mark"], color="teal")
plt.title("Average Overall Mark by Course")
plt.xlabel("Average Mark")
plt.ylabel("Course")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### SQL Operation 4 – Highest Mark in Each Center

Find the maximum overall mark for every center.

In [ ]:
spark.sql("""
SELECT center, MAX(overall_mark) AS highest_mark
FROM students
GROUP BY center
ORDER BY highest_mark DESC
""").show()

## 11. Findings and Challenges

### Findings
- The dataset contains **150,500** student records across 6 courses and 6 regional centers.
- Missing values and duplicate rows were found and cleaned.
- Outliers in overall marks were identified using the IQR method and visualized with a boxplot.
- Most students have **Pass** status.
- Average marks differ across centers and courses (shown in bar charts).
- Some students achieved very high grades (A / A+).

### Challenges
- Setting up PySpark correctly in Google Colab.
- Understanding how to detect and handle missing values, duplicates, and outliers.
- Learning the difference between PySpark DataFrame methods and Spark SQL.
- Creating clear and simple visualizations for better understanding.
- Choosing operations that are easy to explain in the viva.

In [ ]:
spark.stop()
print("Spark stopped.")